In [1]:
import pandas as pd
import numpy as np

#load data 
image_df = pd.read_csv('data/images.csv')

style_df = pd.read_csv('data/styles_FIXED.csv')
style_df = style_df.drop(columns=['Unnamed: 10','Unnamed: 11'])



In [2]:
# importing some more packages 
import requests
import matplotlib.pyplot as plt 

In [3]:
# get only tops, bottoms, and shoes from style df
allowed_vals = ['Topwear', 'Bottomwear', 'Shoes']
filtered_style = style_df[style_df["subCategory"].isin(allowed_vals)]

# separate styles into topwear, bottomwear, and shoes dfs
shoes_style = filtered_style[filtered_style["subCategory"] == "Shoes"]
topWear_style = filtered_style[filtered_style["subCategory"] == "Topwear"]
bottomWear_style = filtered_style[filtered_style["subCategory"] == "Bottomwear"]

In [4]:
# get random samples of each 
# n_samples = 30 
n_samples = 500
samp_btm = bottomWear_style.sample(n=n_samples, random_state=42)
samp_top = topWear_style.sample(n=n_samples, random_state=42)
samp_shoes = shoes_style.sample(n=n_samples, random_state=42)


In [5]:
# remove extensions from filename column 
image_df["filename"]  = image_df["filename"].str.replace(".jpg","")
image_df["filename"]  = pd.to_numeric(image_df["filename"])


In [6]:
#get rows to match with sampled bottomwear from img df
vals = samp_btm["id"].to_list()
samp_btm_imgs = image_df[image_df["filename"].isin(vals)]

#get rows to match with sampled topwear from img df
vals = samp_top["id"].to_list()
samp_top_imgs = image_df[image_df["filename"].isin(vals)]

#get rows to match with sampled shoes from img df
vals = samp_shoes["id"].to_list()
samp_shoes_imgs = image_df[image_df["filename"].isin(vals)]


# concat images to one df
final_imgs_df = pd.concat([samp_btm_imgs, samp_top_imgs, samp_shoes_imgs], ignore_index=True)

# concat styles to one df
final_styles_df = pd.concat([samp_btm, samp_top, samp_shoes], ignore_index=True)

In [7]:
# merge images df with styles df
final_df = pd.merge(final_imgs_df, final_styles_df,left_on = "filename", right_on = "id")
final_df.head()

,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName
0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black
1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts
2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans
3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings
4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants


In [8]:
final_df.shape

(1500, 12)

In [9]:
from PIL import Image
from io import BytesIO
def load_image(url):
    try:
        response = requests.get(url, timeout=10)
        return Image.open(BytesIO(response.content)).convert("RGB")
    except:
        return None

final_df["image_data"] = final_df["link"].apply(load_image)

In [10]:
final_df

,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_data
0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black,<PIL.Image.Image image mode=RGB size=1800x2400...
1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts,<PIL.Image.Image image mode=RGB size=1080x1440...
2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans,<PIL.Image.Image image mode=RGB size=1800x2400...
3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings,<PIL.Image.Image image mode=RGB size=1800x2400...
4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants,<PIL.Image.Image image mode=RGB size=1080x1440...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,17667,http://assets.myntassets.com/v1/images/style/p...,17667,Women,Footwear,Shoes,Casual Shoes,Black,Winter,2015.0,Casual,Catwalk Women Buckel Black Casual Shoe,<PIL.Image.Image image mode=RGB size=1800x2400...
1496,5687,http://assets.myntassets.com/v1/images/style/p...,5687,Men,Footwear,Shoes,Casual Shoes,Brown,Winter,2015.0,Casual,Skechers Men's Traffics Brown Shoe,<PIL.Image.Image image mode=RGB size=1080x1440...
1497,41822,http://assets.myntassets.com/v1/images/style/p...,41822,Women,Footwear,Shoes,Sports Shoes,White,Summer,2012.0,Sports,Skechers Women Grey & White Sports Shoes,<PIL.Image.Image image mode=RGB size=1080x1440...
1498,17694,http://assets.myntassets.com/v1/images/style/p...,17694,Men,Footwear,Shoes,Casual Shoes,Navy Blue,Fall,2011.0,Casual,Vans Men Authentic Navy Blue Casual Shoes,<PIL.Image.Image image mode=RGB size=1800x2400...


In [11]:
final_df.to_csv('/Users/aniyahmcwilliams/Spring 2026/final_df.csv')

In [ ]:
# going to try to pass the images into a basic CNN model 
